
- Consolidates environment setup and installs.
- Defines dataset handling and feature extraction with caching.
- Implements the interaction classifier and training loop.
- Runs evaluation on the dev set and saves the best model.
- Provides an optional multi-view consensus inference + submission generator.

Design notes: the preprocessing and cleaning are written to be language-invariant; users switch a `variant` tag (not a hard language label) via `set_variant()` to point to different dataset folders. 

In [ ]:
# Install required packages (uncomment if needed)
# Note: installing torch in notebooks can be heavy. If you already have a proper env, skip this cell.
# !pip install -q torch torchvision torchaudio sentence-transformers transformers deep-translator

In [ ]:
# Imports and environment detection
import os
import random
import time
import re
import json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sentence_transformers import SentenceTransformer
from PIL import Image

# Optional: CLIP for image features (used if available)
try:
    from transformers import CLIPProcessor, CLIPModel
    _HAS_CLIP = True
except Exception:
    _HAS_CLIP = False

print('CLIP available:', _HAS_CLIP)

In [ ]:
# Seed + device
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# Legacy quick config (matches organizer manual layout)
DATA_ROOT = '/content/dataset'
RUN_FOR = 'china'
TEAM_NAME = 'rms'
PATHS = {
    'train': {'csv': os.path.join(DATA_ROOT, 'train & dev', 'train', 'train.csv'), 'img': os.path.join(DATA_ROOT, 'train & dev', 'train'), 'pt': f'features/{RUN_FOR}_train.pt'},
    'dev':   {'csv': os.path.join(DATA_ROOT, 'train & dev', 'dev', 'dev.csv'),     'img': os.path.join(DATA_ROOT, 'train & dev', 'dev'),   'pt': f'features/{RUN_FOR}_dev.pt'},
    'test':  {'csv': os.path.join(DATA_ROOT, 'test', 'test.csv'),                   'img': os.path.join(DATA_ROOT, 'test'),                 'pt': f'features/{RUN_FOR}_test.pt'},
}
print('Paths configured:', PATHS['train']['csv'])

In [ ]:
# Dataset and feature extraction (caches features in features/*.pt)
class FeatureDataset(Dataset):
    def __init__(self, split_name, text_model_name='sentence-transformers/clip-ViT-B-32-multilingual-v1', clip_model_name='openai/clip-vit-base-patch32'):
        self.split = split_name
        conf = PATHS[split_name]
        self.csv_path = conf['csv']
        self.img_dir = conf['img']
        self.pt_path = conf['pt']

        df = pd.read_csv(self.csv_path)
        self.rows = df.to_dict(orient='records')

        # Try to load cached features
        if os.path.exists(self.pt_path):
            print(f'Loading cached features: {self.pt_path}')
            self.data = torch.load(self.pt_path)
        else:
            print(f'Extracting features for {split_name} (this may take time)')
            # Text model
            t_model = SentenceTransformer(text_model_name)

            # Optional CLIP image model
            if _HAS_CLIP:
                c_model = CLIPModel.from_pretrained(clip_model_name).to(DEVICE)
                c_proc = CLIPProcessor.from_pretrained(clip_model_name)
            else:
                c_model = None
                c_proc = None

            data = []
            for row in tqdm(self.rows, desc=f'extract-{split_name}'):
                img_id = str(row.get('image_id',''))
                if not img_id.lower().endswith(('.jpg', '.png', '.jpeg')):
                    img_id = img_id + '.jpg'
                raw_txt = str(row.get('transcriptions',''))
                clean_txt = re.sub(r'http\S+|www\.\S+|fb\.com\S+|@\w+|#\w+', '', raw_txt).strip()

                # Text vector (CPU)
                T = t_model.encode(clean_txt, convert_to_tensor=True).cpu()

                # Image vector (if CLIP available)
                if c_model is not None and os.path.exists(os.path.join(self.img_dir, img_id)):
                    try:
                        img = Image.open(os.path.join(self.img_dir, img_id)).convert('RGB')
                        v_in = c_proc(images=img, return_tensors='pt')
                        with torch.no_grad():
                            out = c_model.get_image_features(pixel_values=v_in['pixel_values'].to(DEVICE))
                        V = out.cpu().squeeze(0)
                    except Exception as e:
                        V = torch.zeros(T.shape[0])
                else:
                    # fallback: zero vector of size 512 (sentence-transformers clip size)
                    V = torch.zeros(512)

                item = {'id': row.get('image_id'), 'V': V, 'T': T, 'raw': raw_txt}
                if 'original_labels' in row:
                    label_val = str(row.get('original_labels','')).lower().strip()
                    item['y'] = 1.0 if 'not' not in label_val else 0.0
                data.append(item)

            # save cache (CPU-only tensors)
            for it in data:
                it['V'] = it['V'].cpu() if isinstance(it['V'], torch.Tensor) else torch.tensor(it['V'])
                it['T'] = it['T'].cpu() if isinstance(it['T'], torch.Tensor) else torch.tensor(it['T'])
            torch.save(data, self.pt_path)
            print(f'Saved features to {self.pt_path}')
            self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        it = self.data[idx]
        V = it['V'].float()
        T = it['T'].float()
        y = torch.tensor(it['y']).float() if 'y' in it else torch.tensor(-1.0)
        return {'V': V, 'T': T, 'y': y, 'id': it['id'], 'raw': it['raw']}

In [ ]:
# Model definition
class InteractionClassifier(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim*3 + 1, 256), nn.BatchNorm1d(256), nn.GELU(),
            nn.Dropout(0.4), nn.Linear(256, 1)
        )
    def forward(self, V, T):
        V = F.normalize(V, p=2, dim=1)
        T = F.normalize(T, p=2, dim=1)
        inter = V * T
        cos = torch.sum(inter, dim=1, keepdim=True)
        return self.net(torch.cat([V, T, inter, cos], dim=1))

print('Model defined')

In [ ]:
# Configuration — variants + helper to switch dataset variant
# Set DATA_ROOT to the Colab folder that contains 'train & dev' and 'test'.
DEFAULT_DATA_ROOT = '/content/dataset'
TEAM_NAME = 'rms'
# Define supported variants and their label column / output folder mapping
VARIANTS = {
    'china': {'label_col': 'original_labels', 'out_path': 'chinese'},
    'tamil': {'label_col': 'original_labels', 'out_path': 'tamil'},
    'malayalam': {'label_col': 'original_labels', 'out_path': 'malayalam'},
}

def set_variant(variant='china', data_root=DEFAULT_DATA_ROOT):
    """Set the dataset variant (a user-defined tag). This is intentionally language-agnostic.
    Example: set_variant('tamil', '/content/dataset')
    """
    global DATA_ROOT, RUN_FOR, CONFIG, CURR, PATHS
    if variant not in VARIANTS:
        raise ValueError(f'Unknown variant: {variant}. Supported: {list(VARIANTS.keys())}')
    DATA_ROOT = data_root
    RUN_FOR = variant  # `RUN_FOR` is a variant tag, not a hard language label
    CONFIG = VARIANTS
    CURR = CONFIG[RUN_FOR]
    PATHS = {
        'train': {'csv': os.path.join(DATA_ROOT, 'train & dev', 'train', 'train.csv'), 'img': os.path.join(DATA_ROOT, 'train & dev', 'train'), 'pt': f'features/{RUN_FOR}_train.pt'},
        'dev':   {'csv': os.path.join(DATA_ROOT, 'train & dev', 'dev', 'dev.csv'),     'img': os.path.join(DATA_ROOT, 'train & dev', 'dev'),   'pt': f'features/{RUN_FOR}_dev.pt'},
        'test':  {'csv': os.path.join(DATA_ROOT, 'test', 'test.csv'),                   'img': os.path.join(DATA_ROOT, 'test'),                 'pt': f'features/{RUN_FOR}_test.pt'},
    }
    os.makedirs('features', exist_ok=True)
    os.makedirs(f"submission/{CURR['out_path']}", exist_ok=True)
    print(f'Set variant: {RUN_FOR} | data root: {DATA_ROOT}')
    print('Train CSV:', PATHS['train']['csv'])

# Default to 'china' variant using original manual Colab folder layout.
set_variant()

In [ ]:
# Run training (adjust epochs as needed)
# model, best_f1 = train_and_evaluate(epochs=5, batch_size=64, lr=1e-4)
print('To run training: call `train_and_evaluate(epochs=..., batch_size=..., lr=...)`')

In [ ]:
# --- Multi-view consensus inference (test) ---
import time, re, json, logging
logging.basicConfig(level=logging.INFO)

# TRANSLATION: mandatory by design for this approach. We implement caching + retries
TRANSLATION_REQUIRED = True
TRANSLATION_CACHE = f"features/translations_cache_{RUN_FOR}.json"

# Load or create translation cache
if os.path.exists(TRANSLATION_CACHE):
    try:
        with open(TRANSLATION_CACHE, 'r', encoding='utf8') as fh:
            _trans_cache = json.load(fh)
    except Exception:
        _trans_cache = {}
else:
    _trans_cache = {}

if TRANSLATION_REQUIRED:
    try:
        from deep_translator import GoogleTranslator
    except Exception as e:
        raise RuntimeError('deep-translator is required for mandatory translation. Install with `pip install deep-translator`.') from e
    translator = GoogleTranslator(source='auto', target='en')

def safe_translate(text, max_retries=3, backoff=0.5):
    """Translate with caching and retries. Returns original text on persistent failure."""
    if text in _trans_cache:
        return _trans_cache[text]
    for attempt in range(1, max_retries+1):
        try:
            out = translator.translate(text)
            _trans_cache[text] = out
            # persist incrementally to avoid losing results on long runs
            try:
                with open(TRANSLATION_CACHE, 'w', encoding='utf8') as fh:
                    json.dump(_trans_cache, fh, ensure_ascii=False, indent=2)
            except Exception as e:
                logging.warning('Could not save translation cache: %s', e)
            return out
        except Exception as e:
            logging.warning('Translation attempt %s failed: %s', attempt, e)
            time.sleep(backoff * attempt)
    logging.warning('All translation attempts failed; falling back to original text for this item.')
    return text

# Ensure text model for multiview encoding is available
m_model = SentenceTransformer('sentence-transformers/clip-ViT-B-32-multilingual-v1').to(DEVICE)

# Load or use trained model
if 'model' not in globals():
    model = InteractionClassifier().to(DEVICE)
    if os.path.exists(f'best_model_{RUN_FOR}.pth'):
        model.load_state_dict(torch.load(f'best_model_{RUN_FOR}.pth', map_location=DEVICE))
    else:
        raise RuntimeError('No trained model found. Run training first (train_and_evaluate).')
model.eval()

test_ds = FeatureDataset('test')
results = []
for item in tqdm(test_ds, desc='multi-view-predict'):
    V = item['V'].to(DEVICE).unsqueeze(0)
    raw = re.sub(r'http\\S+|www\\.\\S+|fb\\.com\\S+|@\\w+|#\\w+', '', str(item['raw'])).strip()
    v1 = raw
    v2 = ' '.join([w for w in raw.split() if len(w) > 2])
    # v3 is translation (mandatory). safe_translate will fallback to raw if translation fails
    v3 = safe_translate(raw)
    votes = []
    with torch.no_grad():
        for txt in (v1, v2, v3):
            T_v = m_model.encode(txt, convert_to_tensor=True).to(DEVICE).unsqueeze(0)
            votes.append(1 if torch.sigmoid(model(V, T_v)).item() > 0.5 else 0)
    label = 1 if sum(votes) >= 2 else 0
    results.append({'image_id': item['id'], 'label': label, 'votes': votes})

# Save submission (CSV) and a JSON with votes for debugging/analysis
out_csv = f"submission/{CURR['out_path']}/{TEAM_NAME}_taska_{CURR['out_path']}_original.csv"
pd.DataFrame([{'image_id':r['image_id'],'label':r['label']} for r in results]).to_csv(out_csv, index=False)
with open(f"submission/{CURR['out_path']}/{TEAM_NAME}_votes_{CURR['out_path']}.json", 'w', encoding='utf8') as fh:
    json.dump(results, fh, ensure_ascii=False, indent=2)
print('Saved submission:', out_csv)
print('Saved votes debug file:', f"submission/{CURR['out_path']}/{TEAM_NAME}_votes_{CURR['out_path']}.json")